In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd
def split_data(df_clean):

    Y = df_clean['is_fraudulent']
    X = df_clean.drop(columns=['is_fraudulent'], axis=1)   
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.1, random_state=42, stratify=Y)

    # Print dataset sizes
    print(f"Training Set: {X_train.shape[0]} rows")
    print(f"Testing Set: {X_test.shape[0]} rows")  

    return X_train, X_test, y_train, y_test 

df_clean = pd.read_csv('backend_app/training_data/clean_dataset.csv')
X_train, X_test, y_train, y_test = split_data(df_clean)
print(f"X_train : {X_train.shape}")
print(f"X_train : {X_test.shape}")


Training Set: 450000 rows
Testing Set: 50000 rows
X_train : (450000, 7)
X_train : (50000, 7)


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks

2025-02-11 16:52:31.873643: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Define the optimized ANN model
def build_optimized_ann(input_shape):
    model = keras.Sequential([
        # Input layer
        layers.Dense(128, activation='relu', kernel_regularizer=l2(0.001)), 
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # Hidden layers
        layers.Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        # Output layer for binary classification
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Compile model
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Instantiate the model
model = build_optimized_ann(input_shape=X_train.shape[1])

# Add callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=10,  # Increase epochs to allow for more learning
    batch_size=64,  # Adjust batch size for better generalization
    validation_split=0.2,
    shuffle=True,
    callbacks=[early_stopping, reduce_lr]
)

Epoch 1/10
5625/5625 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - accuracy: 0.9884 - loss: 0.1043 - val_accuracy: 1.0000 - val_loss: 0.0056 - learning_rate: 0.0010
Epoch 2/10
5625/5625 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.9995 - loss: 0.0073 - val_accuracy: 1.0000 - val_loss: 0.0033 - learning_rate: 0.0010
Epoch 3/10
5625/5625 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.9996 - loss: 0.0050 - val_accuracy: 1.0000 - val_loss: 0.0029 - learning_rate: 0.0010
Epoch 4/10
5625/5625 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.9998 - loss: 0.0039 - val_accuracy: 1.0000 - val_loss: 0.0039 - learning_rate: 0.0010
Epoch 5/10
5625/5625 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9999 - loss: 0.0032 - val_accuracy: 1.0000 - val_loss: 0.0029 - learning_rate: 0.0010
Epoch 6/10
5625/5625 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9996 - loss: 0.0055 - val_accuracy: 1.0000 - val_loss: 0.0020 - learning_rate: 0.0010
Epoch 7/10
5625/5625 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9

In [7]:
merchant_category = {"groceries" : 0,  'bank' : 1, 'electronics' : 2, 'atm' : 3, 'restaurant' : 4, 'luxury goods' : 5}
        # status = {'success' : 0, 'pending' : 1, 'failed' : 2}
values = {
        'transaction_type': 3,
        'amount': 145.75,  
        'merchant_category': 'groceries',
        'transaction_velocity': 500,  
        'hour': 12,  
        'lon': -102.4194,  
        'transaction_ratio': 0.15247458 
    }
values['merchant_category'] = merchant_category[values['merchant_category']]
values

{'transaction_type': 3,
 'amount': 145.75,
 'merchant_category': 0,
 'transaction_velocity': 500,
 'hour': 12,
 'lon': -102.4194,
 'transaction_ratio': 0.15247458}